In [ ]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm
import gc
import matplotlib.pyplot as plt
import numpy as np
import os
import random
import time
import timm # WICHTIG: pip install timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms.functional as TF  # Das fixiert den NameError
import re
import cv2 # Für das Resizing der Disparity-Map
from torch.cuda.amp import GradScaler, autocast

COCO_CLASSES = [
    'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light',
    'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow',
    'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee',
    'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard',
    'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple',
    'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch',
    'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone',
    'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear',
    'hair drier', 'toothbrush'
]

import os

# Definiere deine Pfade hier:
dataset_paths = {
    # 1. SceneFlow (FlyingThings3D) - Basis für Stereo
    # Erwartet Unterordner wie 'frames_finalpass' und 'disparity'
    'sceneflow': '/home/slarc/datasets/sceneflow"',
    
    # 2. TartanAir (oder UnrealStereo) - Multi-Task & Realism
    # Erwartet Ordner-Struktur des jeweiligen Datensatzes
    'tartanair': '/home/slarc/datasets/TartanAir', 
    
    # 3. COCO (Val2017) - YOLO Finetuning
    # Erwartet direkt den Ordner mit den .jpg Bildern
    'coco': '/home/slarc/datasets/COCO/val2017'
}

# --- Optional: Sicherheits-Check ---
# Prüft, ob die Ordner wirklich da sind, bevor das Training crasht
print("🔍 Prüfe Pfade...")
for key, path in dataset_paths.items():
    if os.path.exists(path):
        print(f"✅ {key}: Gefunden ({path})")
    else:
        print(f"❌ {key}: NICHT GEFUNDEN! ({path})")

# Device Selection (GPU/CPU)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Training auf GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("Training auf CPU (Langsam!)")



In [ ]:
# --- DATASET V9 (Mit Asymmetric Augmentation) ---
class StereoDataset(Dataset):
    def __init__(self, data_dir, mode='train', use_crop=True, use_augmentation=True):
        self.data_dir = data_dir
        self.mode = mode
        self.use_crop = use_crop
        self.use_augmentation = use_augmentation
        
        split = 'train' if mode == 'train' else 'val'
        
        # Pfadsuche
        self.img_root = os.path.join(data_dir, 'FlyingThings3D_subset_image_clean', 'FlyingThings3D_subset', split, 'image_clean')
        if not os.path.exists(self.img_root):
            self.img_root = os.path.join(data_dir, split, 'image_clean')
            
        if not os.path.exists(self.img_root):
             raise ValueError(f"❌ Bild-Ordner nicht gefunden:\n{self.img_root}")

        print(f"[{mode.upper()}] Scanne Bilder in: {self.img_root}")

        self.left_files = []
        self.right_files = []
        self.disp_left_files = []
        self.disp_right_files = []
        
        for root, dirs, files in os.walk(self.img_root):
            for file in files:
                if file.endswith('.png') and 'left' in root:
                    l_path = os.path.join(root, file)
                    r_path = l_path.replace('left', 'right')
                    dl_path = l_path.replace('image_clean', 'disparity').replace('.png', '.pfm')
                    dr_path = r_path.replace('image_clean', 'disparity').replace('.png', '.pfm')
                    
                    if os.path.exists(dl_path):
                        self.left_files.append(l_path)
                        self.right_files.append(r_path)
                        self.disp_left_files.append(dl_path)
                        self.disp_right_files.append(dr_path)
                        
        print(f"[{mode.upper()}] {len(self.left_files)} Paare gefunden.")

    def load_pfm(self, file):
        if not os.path.exists(file): return np.zeros((480, 640), dtype=np.float32)
        with open(file, "rb") as f:
            header = f.readline().decode('utf-8').rstrip()
            if header == 'PF': color = True
            elif header == 'Pf': color = False
            else: raise Exception('Keine PFM Datei.')

            dims = f.readline().decode('utf-8').split()
            width = int(dims[0])
            height = int(dims[1])

            scale = float(f.readline().decode('utf-8').rstrip())
            if scale < 0:
                endian = '<'
                scale = -scale
            else:
                endian = '>'

            data = np.fromfile(f, endian + 'f')
            shape = (height, width, 3) if color else (height, width)

            data = np.reshape(data, shape)
            data = np.flipud(data)
            
            # --- FIX ---
            # 1. NaN/Inf bereinigen
            data = np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)
            
            # 2. Absolutwert nehmen! 
            # Deine Daten sind negativ (-79 bis -1), das muss positiv werden.
            data = np.abs(data)
            # -----------

            return data.copy()

    def __len__(self):
        return len(self.left_files)

    def __getitem__(self, idx):
        l_path = self.left_files[idx]
        r_path = self.right_files[idx]
        
        left = Image.open(l_path).convert('L')
        right = Image.open(r_path).convert('L')
        dl = self.load_pfm(self.disp_left_files[idx])
        dr = self.load_pfm(self.disp_right_files[idx])
        
        orig_w, orig_h = left.size
        target_w, target_h = 640, 480
        
        left = left.resize((target_w, target_h), Image.BILINEAR)
        right = right.resize((target_w, target_h), Image.BILINEAR)
        scale_x = target_w / orig_w
        dl = cv2.resize(dl, (target_w, target_h), interpolation=cv2.INTER_LINEAR) * scale_x
        dr = cv2.resize(dr, (target_w, target_h), interpolation=cv2.INTER_LINEAR) * scale_x

        l_np = np.array(left, dtype=np.float32) / 255.0
        r_np = np.array(right, dtype=np.float32) / 255.0
        dl_np = np.ascontiguousarray(dl, dtype=np.float32)
        dr_np = np.ascontiguousarray(dr, dtype=np.float32)

        if self.mode == 'train' and self.use_crop:
            crop_h, crop_w = 320, 640
            y = random.randint(0, target_h - crop_h)
            x = random.randint(0, target_w - crop_w)
            
            l_np = l_np[y:y+crop_h, x:x+crop_w]
            r_np = r_np[y:y+crop_h, x:x+crop_w]
            dl_np = dl_np[y:y+crop_h, x:x+crop_w]
            dr_np = dr_np[y:y+crop_h, x:x+crop_w]
            
            if self.use_augmentation:
                def augment_photo(img):
                    mult = 0.8 + np.random.rand() * 0.4 
                    img = img * mult
                    mean = img.mean()
                    contrast = 0.8 + np.random.rand() * 0.4
                    img = (img - mean) * contrast + mean
                    img = np.clip(img, 0, 1)
                    gamma = 0.8 + np.random.rand() * 0.4
                    img = img ** gamma
                    return np.clip(img, 0, 1)

                l_np = augment_photo(l_np)
                r_np = augment_photo(r_np)

        l_t = torch.from_numpy(l_np).unsqueeze(0)
        r_t = torch.from_numpy(r_np).unsqueeze(0)
        dl_t = torch.from_numpy(dl_np).unsqueeze(0)
        dr_t = torch.from_numpy(dr_np).unsqueeze(0)
        
        return l_t, r_t, dl_t, dr_t

In [ ]:
import os
import glob
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from ultralytics import YOLO
import torchvision

# --- A. TARTAN AIR DATASET (Phase 2) ---
class TartanAirDataset(Dataset):
    def __init__(self, data_dir, mode='train', target_size=(480, 640)):
        self.files = []
        self.target_h, self.target_w = target_size
        
        # Rekursiv nach Bildern und Depth suchen
        for root, dirs, files in os.walk(data_dir):
            if 'image_left' in dirs and 'depth_left' in dirs:
                img_dir = os.path.join(root, 'image_left')
                depth_dir = os.path.join(root, 'depth_left')
                imgs = sorted(glob.glob(os.path.join(img_dir, "*.png")))
                
                for img_path in imgs:
                    fname = os.path.basename(img_path)
                    # Depth file pattern anpassen falls nötig (oft _left_depth.npy)
                    depth_name = fname.replace("_left.png", "_left_depth.npy")
                    depth_path = os.path.join(depth_dir, depth_name)
                    right_path = img_path.replace("image_left", "image_right").replace("_left", "_right")
                    
                    if os.path.exists(depth_path) and os.path.exists(right_path):
                        self.files.append((img_path, right_path, depth_path))
        print(f"[{mode}] TartanAir: {len(self.files)} Paare gefunden.")

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        l_path, r_path, d_path = self.files[idx]
        left = Image.open(l_path).convert('L')
        right = Image.open(r_path).convert('L')
        depth_np = np.load(d_path) 
        
        # Depth -> Disp (TartanAir: fx=320, B=0.25)
        with np.errstate(divide='ignore'):
            disp_np = (320.0 * 0.25) / (depth_np + 1e-6)
            
        # Resize & Norm
        left = left.resize((self.target_w, self.target_h), Image.BILINEAR)
        right = right.resize((self.target_w, self.target_h), Image.BILINEAR)
        disp_pil = Image.fromarray(disp_np)
        disp_pil = disp_pil.resize((self.target_w, self.target_h), Image.NEAREST)
        disp_np = np.array(disp_pil) * (self.target_w / 640.0) # Scale Disp Values

        t = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean=[0.5], std=[0.5])])
        return t(left), t(right), torch.from_numpy(disp_np).float().unsqueeze(0)

# --- B. COCO DATASET (Phase 3) ---
class COCOSimpleDataset(Dataset):
    def __init__(self, data_dir, target_size=(480, 640)):
        self.files = sorted(glob.glob(os.path.join(data_dir, "*.jpg")))
        self.target_h, self.target_w = target_size
        print(f"[COCO] Found {len(self.files)} images for YOLO Finetuning.")

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert('L')
        img = img.resize((self.target_w, self.target_h), Image.BILINEAR)
        t = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean=[0.5], std=[0.5])])
        # Dummy Stereo Data
        return t(img), t(img), torch.zeros((1, self.target_h, self.target_w))

# --- C. TEACHER CLASS (Distillation) ---
class DistillationTeachers(nn.Module):
    def __init__(self, device):
        super().__init__()
        self.device = device
        print("👨‍🏫 Loading Teachers...")
        self.seg_teacher = torchvision.models.segmentation.lraspp_mobilenet_v3_large(weights='DEFAULT').to(device).eval()
        self.yolo_teacher = YOLO('yolov8n.pt')
        self.yolo_model = self.yolo_teacher.model.to(device).eval()
        
        # Mapping Cityscapes -> Custom (0:Floor, 1:Static, 2:Dynamic, 3:Void)
        self.label_map = torch.zeros(20, dtype=torch.long, device=device) + 3
        self.label_map[[0, 1, 9]] = 0 # Road, Sidewalk, Terrain
        self.label_map[[2, 3, 4, 5, 6, 7, 8]] = 1 # Static Objects
        self.label_map[[11, 12, 13, 14, 15, 16, 17, 18]] = 2 # Vehicles/Humans

    @torch.no_grad()
    def generate_targets(self, img_tensor):
        # ... (RGB Konvertierung wie vorher) ...
        img_rgb = (img_tensor * 0.5 + 0.5).repeat(1, 3, 1, 1)

        # 1. Seg Targets (Bleibt gleich)
        seg_preds = torch.argmax(self.seg_teacher(img_rgb)['out'], dim=1)
        seg_targets = self.label_map[seg_preds]
        
        # 2. YOLO Targets (Objectness + Class ID)
        target_h, target_w = img_tensor.shape[2] // 32, img_tensor.shape[3] // 32
        
        # A. Objectness Heatmap [B, 1, H, W]
        batch_heatmaps = torch.zeros((img_tensor.shape[0], 1, target_h, target_w), device=self.device)
        
        # B. Class ID Map [B, H, W] (Initialisiert mit -1 als "Kein Objekt")
        batch_classes = torch.full((img_tensor.shape[0], target_h, target_w), -1, dtype=torch.long, device=self.device)
        
        results = self.yolo_teacher(img_rgb, verbose=False)
        
        for i, res in enumerate(results):
            # Iteriere über gefundene Boxen
            for box, cls in zip(res.boxes.xywhn, res.boxes.cls):
                # Zentrum der Box im Grid berechnen
                cx, cy = int(box[0]*target_w), int(box[1]*target_h)
                
                if 0 <= cx < target_w and 0 <= cy < target_h:
                    # Objectness setzen
                    batch_heatmaps[i, 0, cy, cx] = 1.0
                    
                    # Class ID setzen (z.B. 0=Person, 56=Chair)
                    batch_classes[i, cy, cx] = int(cls.item())
                    
        return seg_targets, batch_heatmaps, batch_classes

In [ ]:
import os
import glob
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class UnrealStereoDataset(Dataset):
    def __init__(self, data_dir, mode='train', target_size=(480, 640)):
        self.data_dir = data_dir
        self.mode = mode
        self.target_h, self.target_w = target_size
        self.file_pairs = []

        # 1. Ordner Scannen (00000 bis 00008)
        # Wir nutzen alle verfügbaren Ordner
        sequences = sorted([d for d in os.listdir(data_dir) if d.isdigit() and len(d)==5])
        
        # Split: Letzter Ordner ist Val, Rest Train (außer im Finisher Mode, da alles Train)
        if mode == 'val':
            sequences = sequences[-1:]
        elif mode == 'train':
            sequences = sequences[:-1]
        
        print(f"[{mode.upper()}] Unreal Dataset scannt Sequenzen: {sequences}")

        for seq in sequences:
            # Pfade hardcoded auf deine Struktur
            path_img0 = os.path.join(data_dir, seq, "Image0")
            path_img1 = os.path.join(data_dir, seq, "Image1")
            path_disp = os.path.join(data_dir, seq, "Disp0")
            
            # Alle Bilder holen
            images = sorted(glob.glob(os.path.join(path_img0, "*.jpg")))
            
            for img_path in images:
                fname = os.path.basename(img_path)
                left = img_path
                right = os.path.join(path_img1, fname)
                disp = os.path.join(path_disp, fname.replace(".jpg", ".npy"))
                
                if os.path.exists(right) and os.path.exists(disp):
                    self.file_pairs.append((left, right, disp))
        
        print(f"[{mode.upper()}] Unreal: {len(self.file_pairs)} Paare gefunden.")

    def __len__(self):
        return len(self.file_pairs)

    def __getitem__(self, idx):
        l_path, r_path, d_path = self.file_pairs[idx]
        
        # Laden (Grayscale für dein Modell!)
        left = Image.open(l_path).convert('L')
        right = Image.open(r_path).convert('L')
        
        # Disparität laden
        disp_np = np.load(d_path) # Shape oft [H, W] oder [H, W, 1]
        if disp_np.ndim == 3: disp_np = disp_np[:,:,0]
        
        # Resize Logic
        w_orig, h_orig = left.size
        
        left = left.resize((self.target_w, self.target_h), Image.BILINEAR)
        right = right.resize((self.target_w, self.target_h), Image.BILINEAR)
        
        # Disp Resize & Value Scaling
        disp_pil = Image.fromarray(disp_np)
        disp_pil = disp_pil.resize((self.target_w, self.target_h), Image.NEAREST)
        disp_np = np.array(disp_pil)
        
        # WICHTIG: Disparität mitskalieren!
        scale_x = self.target_w / w_orig
        disp_np = disp_np * scale_x
        
        # Transform
        t = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5], std=[0.5])
        ])
        
        return t(left), t(right), torch.from_numpy(disp_np).float().unsqueeze(0)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

# --- Helper Classes (Standard) ---
class Conv2dReLU6(nn.Module):
    def __init__(self, in_c, out_c, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.conv = nn.Conv2d(in_c, out_c, kernel_size, stride, padding, bias=False)
        self.bn = nn.BatchNorm2d(out_c)
        self.act = nn.ReLU6(inplace=True)
    def forward(self, x): return self.act(self.bn(self.conv(x)))

class DepthwiseSeparable(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.depthwise = nn.Conv2d(in_c, in_c, 3, padding=1, groups=in_c, bias=False)
        self.pointwise = nn.Conv2d(in_c, out_c, 1, bias=False)
        self.bn = nn.BatchNorm2d(out_c)
        self.act = nn.ReLU6(inplace=True)
    def forward(self, x): return self.act(self.bn(self.pointwise(x)))

class StructureBlock(nn.Module):
    def __init__(self):
        super().__init__()
        sobel_x = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]]).view(1, 1, 3, 3)
        sobel_y = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]]).view(1, 1, 3, 3)
        self.register_buffer('k_sx', sobel_x)
        self.register_buffer('k_sy', sobel_y)
    def forward(self, x):
        sx = F.conv2d(x, self.k_sx, padding=1)
        sy = F.conv2d(x, self.k_sy, padding=1)
        return torch.abs(sx) + torch.abs(sy)

# --- NPU Friendly Occlusion Head ---
class NPUOcclusionHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = Conv2dReLU6(3, 32, 3, 1, 1)
        self.conv2 = Conv2dReLU6(32, 16, 3, 1, 1)
        self.out = nn.Conv2d(16, 1, 1)
        nn.init.zeros_(self.out.bias)
    def forward(self, disp, structure, confidence):
        x = torch.cat([disp, structure, confidence], dim=1)
        x = self.conv1(x)
        x = self.conv2(x)
        return self.out(x)

class MiniUNetRefiner(nn.Module):
    def __init__(self, in_channels, max_disp=192, edge_in_channels=2):
        super().__init__()
        self.max_disp = max_disp
        self.enc1 = Conv2dReLU6(in_channels, 32)
        self.down1 = Conv2dReLU6(32, 32, stride=2)
        self.enc2 = Conv2dReLU6(32, 48) 
        self.down2 = Conv2dReLU6(48, 48, stride=2)
        
        self.center = nn.Sequential(
            Conv2dReLU6(48, 48), 
            DepthwiseSeparable(48, 48)
        )
        
        self.up2 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.dec2 = Conv2dReLU6(48 + 48, 32)
        self.up1 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.dec1 = Conv2dReLU6(32 + 32, 16)
        
        self.final = nn.Conv2d(16, 1, kernel_size=3, padding=1)
        self.edge_refine = nn.Sequential(
            nn.Conv2d(edge_in_channels, 16, 3, padding=1), 
            nn.ReLU6(inplace=True),
            nn.Conv2d(16, 1, 3, padding=1)
        )
        nn.init.uniform_(self.final.weight, -0.01, 0.01)
        nn.init.uniform_(self.edge_refine[-1].weight, -0.001, 0.001)

    def forward(self, disp_curr, features, structure):
        disp_norm = disp_curr / self.max_disp
        x = torch.cat([disp_norm, features, structure], dim=1)
        e1 = self.enc1(x)
        e2 = self.enc2(self.down1(e1))
        c = self.center(self.down2(e2))
        d2 = self.dec2(torch.cat([self.up2(c), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        res_main = self.final(d1)
        res_edge = self.edge_refine(torch.cat([disp_norm, structure], dim=1))
        if disp_curr.shape[1] > 1: disp_base = disp_curr[:, 0:1, :, :]
        else: disp_base = disp_curr
        out = disp_base + 0.7 * (res_main + res_edge)
        return torch.clamp(out, 0, self.max_disp)

# --- NEU: YOLO HEAD (Shared Backbone) ---
class SharedYOLOHead(nn.Module):
    """ Bereitet für YOLOv8 Nano vor (Anpassung der Channels) """
    def __init__(self, channels_list=[40, 112, 960], num_classes=80):
        super().__init__()
        # P3(Stride8), P4(Stride16), P5(Stride32) aus MobileNetV3-Large
        # Wir adaptieren auf eine gemeinsame Breite (z.B. 64 für Nano-Scale)
        hidden_dim = 64
        self.adapter_p3 = nn.Conv2d(channels_list[0], hidden_dim, 1)
        self.adapter_p4 = nn.Conv2d(channels_list[1], hidden_dim, 1)
        self.adapter_p5 = nn.Conv2d(channels_list[2], hidden_dim, 1)
        
        # Einfacher FPN-Lite Block (Feature Fusion)
        self.up = nn.Upsample(scale_factor=2, mode='nearest')
        self.fuse_p4 = Conv2dReLU6(hidden_dim, hidden_dim)
        self.fuse_p3 = Conv2dReLU6(hidden_dim, hidden_dim)
        
        # Detection Heads (Sehr simpel für den Anfang)
        # Output: 4 Box-Coords + 1 Objectness + N Classes
        self.head_p3 = nn.Conv2d(hidden_dim, 4 + 1 + num_classes, 1)
        self.head_p4 = nn.Conv2d(hidden_dim, 4 + 1 + num_classes, 1)
        self.head_p5 = nn.Conv2d(hidden_dim, 4 + 1 + num_classes, 1)

    def forward(self, x_p3, x_p4, x_p5):
        # 1. Adaptieren
        f3 = self.adapter_p3(x_p3)
        f4 = self.adapter_p4(x_p4)
        f5 = self.adapter_p5(x_p5)
        
        # 2. Top-Down Path (P5 -> P4 -> P3)
        f4 = self.fuse_p4(f4 + self.up(f5))
        f3 = self.fuse_p3(f3 + self.up(f4))
        
        # 3. Predict
        out3 = self.head_p3(f3) # Small Objects
        out4 = self.head_p4(f4) # Medium Objects
        out5 = self.head_p5(f5) # Large Objects
        
        return [out3, out4, out5]

# --- NEU: SEGMENTATION HEAD (LR-ASPP) ---
class LiteRASPPHead(nn.Module):
    """ LR-ASPP für Segmentation (Boden/Wand) """
    def __init__(self, low_channels=40, high_channels=960, num_classes=4):
        super().__init__()
        self.cbr = nn.Sequential(
            nn.Conv2d(high_channels, 128, 1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True)
        )
        self.scale = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(high_channels, 128, 1, bias=False), nn.Sigmoid()
        )
        self.low_classifier = nn.Conv2d(low_channels, num_classes, 1)
        self.high_classifier = nn.Conv2d(128, num_classes, 1)

    def forward(self, x_low, x_high):
        # x_low: Stride 8 (P3), x_high: Stride 16 oder 32 (P5)
        x = self.cbr(x_high)
        s = self.scale(x_high)
        x = x * s
        x = F.interpolate(x, size=x_low.shape[-2:], mode='bilinear', align_corners=False)
        return self.high_classifier(x) + self.low_classifier(x_low)


# --- HAUPTKLASSE V10 (MULTI-TASK) ---
class StereoNet_MultiTask_V10(nn.Module):
    def __init__(self, max_disp=192):
        super().__init__()
        self.max_disp = max_disp
        self.groups = 12 
        self.softmax_temp = nn.Parameter(torch.tensor(1.0))
        
        # 1. Backbone (Gray) - Wir brauchen P3, P4, P5
        # Indices: 1=Stride4, 2=Stride8(P3), 3=Stride16(P4), 4=Stride32(P5)
        self.backbone = timm.create_model('mobilenetv3_large_100', pretrained=True, 
                                          features_only=True, out_indices=(1, 2, 3, 4), in_chans=1)
        
        # Stereo Adapters (Nutzen Stride 4 und Stride 8)
        self.lat_s4 = nn.Conv2d(24, 32, 1, bias=False)  # Index 1
        self.lat_s8 = nn.Conv2d(40, 32, 1, bias=False)  # Index 2
        
        # Global Context (Nutzt Stride 8 für Stereo)
        self.global_context = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(40, 32, 1, bias=False),
            nn.ReLU6(inplace=True)
        )

        self.fuse = Conv2dReLU6(32, 32, 3, 1, 1)
        self.feat_norm = nn.GroupNorm(8, 32)
        self.structure = StructureBlock()
        
        # Stereo Cost Volume & Filter
        self.cost_reducer = nn.Sequential(
            nn.Conv2d(96, 32, kernel_size=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU6(inplace=True),
            nn.Conv2d(32, self.groups, kernel_size=1, bias=False)
        )
        self.cost_pre_filter = DepthwiseSeparable(self.groups * (max_disp // 4), self.groups * (max_disp // 4))
        self.spatial_filter = nn.Sequential(
            DepthwiseSeparable(self.groups * (max_disp // 4), self.groups * (max_disp // 4)),
            nn.Conv2d(self.groups * (max_disp // 4), max_disp // 4, kernel_size=1, bias=False)
        )
        
        # Refiners
        self.refine_low = MiniUNetRefiner(34, max_disp, edge_in_channels=2)
        self.refine_v1  = MiniUNetRefiner(34, max_disp, edge_in_channels=2)
        self.refine_final = MiniUNetRefiner(35, max_disp, edge_in_channels=3)
        self.occ_head = NPUOcclusionHead()

        # --- NEW: AUXILIARY TASK HEADS ---
        # Channels aus MobileNetV3 Large (100):
        # Idx 2 (P3/Stride 8): 40 Ch
        # Idx 3 (P4/Stride 16): 112 Ch
        # Idx 4 (P5/Stride 32): 960 Ch
        self.seg_head = LiteRASPPHead(low_channels=40, high_channels=960, num_classes=4)
        self.yolo_head = SharedYOLOHead(channels_list=[40, 112, 960], num_classes=80)
        
        self.enable_aux_tasks = False # Default: Aus (nur Stereo Training)

    def build_cost_volume_fast(self, feat_l, feat_r, max_disp_4):
        B, C, H, W = feat_l.shape
        feat_r_padded = F.pad(feat_r, (max_disp_4 - 1, 0, 0, 0))
        feat_r_unfolded = feat_r_padded.unfold(3, W, 1).permute(0, 3, 1, 2, 4)
        feat_l_exp = feat_l.unsqueeze(1).expand(-1, max_disp_4, -1, -1, -1)
        diff = feat_l_exp - feat_r_unfolded
        return torch.cat([feat_l_exp, feat_r_unfolded, diff], dim=2)

    def forward_single(self, left, right):
        # 1. Feature Extraction (Shared Backbone)
        # Returns list: [stride4(24), stride8(40), stride16(112), stride32(960)]
        feats_l_all = self.backbone(left)
        
        # Features für Aux Tasks sichern (Links)
        p3_l = feats_l_all[1] # Stride 8
        p4_l = feats_l_all[2] # Stride 16
        p5_l = feats_l_all[3] # Stride 32 (für Seg & Yolo)

        # Features für Stereo (Links)
        f4_l = self.lat_s4(feats_l_all[0]) # Stride 4
        f8_l = self.lat_s8(p3_l)           # Stride 8
        g_l = self.global_context(p3_l)
        
        # Stereo Fusion Links
        f8_l_up = F.interpolate(f8_l, scale_factor=2, mode='bilinear', align_corners=False)
        feat_l = self.fuse(f4_l + f8_l_up + g_l)
        feat_l = F.normalize(self.feat_norm(feat_l), dim=1)
        
        # --- AUX TASKS (YOLO & SEG) ---
        seg_out = None
        yolo_out = None
        if self.enable_aux_tasks:
            # Segmentation (Boden/Wand): Nutzt P3 (Detail) und P5 (Kontext)
            seg_out = self.seg_head(p3_l, p5_l)
            
            # YOLO (Objekte): Nutzt P3, P4, P5
            yolo_out = self.yolo_head(p3_l, p4_l, p5_l)

        # --- STEREO RECHTS ---
        # Nur nötig, wenn wir Stereo berechnen
        feats_r_all = self.backbone(right)
        f4_r = self.lat_s4(feats_r_all[0])
        f8_r = self.lat_s8(feats_r_all[1])
        g_r = self.global_context(feats_r_all[1])
        f8_r_up = F.interpolate(f8_r, scale_factor=2, mode='bilinear', align_corners=False)
        feat_r = self.fuse(f4_r + f8_r_up + g_r)
        feat_r = F.normalize(self.feat_norm(feat_r), dim=1)

        # Structure
        struct_l = self.structure(left)

        # 2. Cost Volume & Stereo Pipeline
        B, C, H4, W4 = feat_l.shape
        max_disp_4 = self.max_disp // 4
        
        cost_vol = self.build_cost_volume_fast(feat_l, feat_r, max_disp_4)
        cost_red = self.cost_reducer(cost_vol.reshape(B * max_disp_4, 96, H4, W4))
        cost_red = cost_red.view(B, max_disp_4 * self.groups, H4, W4)
        cost_red = self.cost_pre_filter(cost_red)
        cost_out = self.spatial_filter(cost_red)
        
        temp = torch.clamp(self.softmax_temp, 0.5, 2.0)
        prob = F.softmax(-cost_out * temp, dim=1)
        confidence, _ = torch.max(prob, dim=1, keepdim=True)
        
        d_range = torch.arange(max_disp_4, device=left.device).view(1, -1, 1, 1).float()
        disp_4 = torch.sum(prob * d_range, dim=1, keepdim=True)
        
        # Refinement Loop
        struct_4 = F.interpolate(struct_l, scale_factor=0.25, mode='area')
        disp_4_scaled = disp_4 * 4.0
        disp_low_ref = self.refine_low(disp_4_scaled, feat_l, struct_4)
        
        disp_2 = F.interpolate(disp_low_ref, scale_factor=2, mode='bilinear', align_corners=False)
        feat_2 = F.interpolate(feat_l, scale_factor=2, mode='bilinear', align_corners=False)
        struct_2 = F.interpolate(struct_l, scale_factor=0.5, mode='area')
        disp_v1_ref = self.refine_v1(disp_2, feat_2, struct_2)
        
        disp_1 = F.interpolate(disp_v1_ref, scale_factor=2, mode='bilinear', align_corners=False)
        feat_1 = F.interpolate(feat_l, scale_factor=4, mode='bilinear', align_corners=False)
        conf_1 = F.interpolate(confidence, scale_factor=4, mode='bilinear', align_corners=False)
        
        occ_logits = self.occ_head(disp_1, struct_l, conf_1)
        occ_prob = torch.sigmoid(occ_logits)
        
        disp_final = self.refine_final(torch.cat([disp_1, occ_prob], dim=1), feat_1, struct_l)
        
        # Return Tuple mit Aux Outputs
        return disp_final, disp_v1_ref, disp_low_ref, occ_logits, seg_out, yolo_out

    def forward(self, left, right):
        # Wrapper für PyTorch Standard-Calls
        return self.forward_single(left, right)

In [ ]:
@torch.no_grad()
def validate(model, val_loader, device):
    model.eval()
    
    total_epe = 0.0
    total_loss = 0.0
    valid_batches = 0
    
    # tqdm für Fortschrittsbalken
    pbar = tqdm(val_loader, desc="🔍 Validierung", leave=False, ncols=150)
    
    # FIX: Jetzt 4 Werte entpacken statt 3
    for left, right, gt_L, gt_R in pbar:
        left, right = left.to(device), right.to(device)
        gt_L = gt_L.to(device)
        # gt_R brauchen wir für EPE-Validierung eigentlich nicht zwingend, 
        # aber wir müssen es entpacken, damit Python nicht meckert.

        # Forward Pass (Wir nutzen nur LR Core für Speed)
        # model.core gibt zurück: (disp_final, disp_v1, disp_low, occ)
        out = model.core(left, right)
        disp_pred = out[0] # Wir nehmen nur die finale Disparität
        
        # Validitäts-Maske (Nur Pixel prüfen, die Ground Truth haben)
        mask = (gt_L > 0) & (gt_L < 192)
        
        if mask.sum() > 0:
            # 1. EPE (End Point Error) berechnen
            # Absoluter Abstand in Pixeln
            diff = torch.abs(disp_pred[mask] - gt_L[mask])
            epe = diff.mean().item()
            
            # 2. Loss berechnen (Smooth L1 als Referenz)
            loss = F.smooth_l1_loss(disp_pred[mask], gt_L[mask], beta=1.0).item()
            
            total_epe += epe
            total_loss += loss
            valid_batches += 1
            
            pbar.set_postfix({'val_epe': f"{epe:.2f}"})

    if valid_batches == 0:
        return 0.0, 0.0

    return (total_loss / valid_batches), (total_epe / valid_batches)


In [ ]:
# --- LOSSES V9.3 (Edge-Aware & Adaptive) ---

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# --- 1. SSIM (Benötigt für den Loss, muss definiert sein) ---
def get_ssim_window(window_size, channel):
    def gaussian(window_size, sigma):
        gauss = torch.Tensor([np.exp(-(x - window_size//2)**2/float(2*sigma**2)) for x in range(window_size)])
        return gauss/gauss.sum()
    _1D_window = gaussian(window_size, 1.5).unsqueeze(1)
    _2D_window = _1D_window.mm(_1D_window.t()).float().unsqueeze(0).unsqueeze(0)
    window = _2D_window.expand(channel, 1, window_size, window_size).contiguous()
    return window

class SSIM(nn.Module):
    def __init__(self, window_size=11, channel=1):
        super(SSIM, self).__init__()
        self.window_size = window_size
        self.channel = channel
        self.register_buffer('window', get_ssim_window(window_size, channel))
    def forward(self, img1, img2):
        mu1 = F.conv2d(img1, self.window, padding=self.window_size//2, groups=self.channel)
        mu2 = F.conv2d(img2, self.window, padding=self.window_size//2, groups=self.channel)
        mu1_sq, mu2_sq, mu1_mu2 = mu1.pow(2), mu2.pow(2), mu1*mu2
        sigma1_sq = F.conv2d(img1*img1, self.window, padding=self.window_size//2, groups=self.channel) - mu1_sq
        sigma2_sq = F.conv2d(img2*img2, self.window, padding=self.window_size//2, groups=self.channel) - mu2_sq
        sigma12 = F.conv2d(img1*img2, self.window, padding=self.window_size//2, groups=self.channel) - mu1_mu2
        C1, C2 = 0.01**2, 0.03**2
        ssim_map = ((2*mu1_mu2 + C1)*(2*sigma12 + C2))/((mu1_sq + mu2_sq + C1)*(sigma1_sq + sigma2_sq + C2))
        return ssim_map.mean()

# --- 2. Charbonnier ---
def charbonnier_loss(x, y, eps=1e-3):
    return torch.sqrt((x - y)**2 + eps**2).mean()

# --- 3. Smoothness (Safe & Adaptive) ---
def smoothness_loss_adaptive(pred_disp, img, beta=9.0):
    def gradient_x(x): return F.pad(x, (0, 1, 0, 0))[:, :, :, 1:] - x
    def gradient_y(x): return F.pad(x, (0, 0, 0, 1))[:, :, 1:, :] - x
    
    disp_gradients_x = gradient_x(pred_disp)
    disp_gradients_y = gradient_y(pred_disp)

    image_gradients_x = gradient_x(img)
    image_gradients_y = gradient_y(img)

    weights_x = torch.exp(-torch.mean(torch.abs(image_gradients_x), 1, keepdim=True) * beta)
    weights_y = torch.exp(-torch.mean(torch.abs(image_gradients_y), 1, keepdim=True) * beta)

    smoothness_x = torch.abs(disp_gradients_x) * weights_x
    smoothness_y = torch.abs(disp_gradients_y) * weights_y
    return (smoothness_x + smoothness_y).mean()

# --- HAUPTFUNKTION V9.3 ---
def robust_stereo_loss_v9(outputs, left_img, right_img, gt_disp_L, gt_disp_R=None, 
                           w_geom=0.8, w_photo=1.0, w_lrc=0.5, w_smooth=0.1, w_occ=0.2):
    """
    Version 9.3 (Edge-Aware & Adaptive):
    - Integriert Edge-Weighted Photometric Loss
    - Integriert Adaptive Scale Weights
    - Integriert Dynamic LRC Threshold
    - Integriert Masked Smoothness Loss
    """
    
    # --- Initialisierung ---
    if not hasattr(robust_stereo_loss_v9, 'ssim_module'):
        robust_stereo_loss_v9.ssim_module = SSIM(channel=1).to(left_img.device)
    ssim_loss_fn = robust_stereo_loss_v9.ssim_module
    
    # V9.3 Change: Angepasste Scale Weights
    scale_weights = [1.0, 0.7, 0.3] 
    total_loss, logs = 0.0, {}
    
    _, _, H_full, W_full = left_img.shape

    # --- Helper für Gradienten und Magnitude ---
    def get_gradients(img):
        gx = F.pad(img, (0, 1, 0, 0))[:, :, :, 1:] - img
        gy = F.pad(img, (0, 0, 0, 1))[:, :, 1:, :] - img
        return torch.abs(gx) + torch.abs(gy)

    def get_edge_magnitude(img):
        grads = get_gradients(img)
        return grads.mean(dim=1, keepdim=True)

    # --- Multi-Scale-Loop ---
    for i, weight in enumerate(scale_weights):
        # Aktuelle Prediction holen
        disp_L = outputs["LR"][i]
        disp_R = outputs["RL"][i] if "RL" in outputs else None

        # Aktuelle Auflösung
        _, _, H_curr, W_curr = disp_L.shape

        # --- Scale Adjustments ---
        if W_curr != W_full:
            scale_factor = W_curr / W_full
            # GT skalieren
            gt_L_curr = F.interpolate(gt_disp_L, size=(H_curr, W_curr), mode='nearest-exact') * scale_factor
            # Bilder skalieren
            img_L_curr = F.interpolate(left_img, size=(H_curr, W_curr), mode='bilinear', align_corners=False)
            img_R_curr = F.interpolate(right_img, size=(H_curr, W_curr), mode='bilinear', align_corners=False)
        else:
            gt_L_curr = gt_disp_L
            img_L_curr, img_R_curr = left_img, right_img
            
        # Nan-Schutz
        disp_L = torch.nan_to_num(disp_L, nan=0.0, posinf=192.0, neginf=0.0)
            
        # --- 1. Geometrischer Loss (Supervised) ---
        if gt_L_curr is not None:
            mask_valid = (gt_L_curr > 0) & (gt_L_curr < 192)
            if mask_valid.sum() > 0:
                loss_g = charbonnier_loss(disp_L[mask_valid], gt_L_curr[mask_valid])
                total_loss += w_geom * weight * loss_g
                if i == 0: logs['geom'] = loss_g.item()

        # --- 2. Self-Supervised Losses ---
        if disp_R is not None:
            # Grid
            grid_y_curr, grid_x_curr = torch.meshgrid(torch.arange(H_curr, device=left_img.device), 
                                                      torch.arange(W_curr, device=left_img.device), indexing='ij')
            grid_x_norm = (2.0 * grid_x_curr / (W_curr - 1)) - 1.0
            grid_y_norm = (2.0 * grid_y_curr / (H_curr - 1)) - 1.0
            
            # Warping
            disp_L_norm = 2.0 * disp_L / (W_curr - 1)
            vgrid = torch.stack((grid_x_norm - disp_L_norm.squeeze(1), 
                                 grid_y_norm.expand(disp_L.shape[0], H_curr, W_curr)), dim=3)
            
            disp_R_warped = F.grid_sample(disp_R, vgrid, align_corners=False, padding_mode='border')
            right_warped = F.grid_sample(img_R_curr, vgrid, align_corners=False, padding_mode='border')
            
            # --- LRC Loss ---
            lrc_diff = torch.abs(disp_L - disp_R_warped)
            loss_l = charbonnier_loss(lrc_diff, torch.zeros_like(lrc_diff))
            total_loss += w_lrc * weight * loss_l
            if i == 0: logs['lrc'] = loss_l.item()

            # --- V9.3: Dynamic Threshold & Edge Weights ---
            
            # 1. Dynamic Threshold
            thr_lrc = 1.2 * (W_curr / W_full) # Skaliert mit Auflösung
            mask_vis = (lrc_diff < thr_lrc + 0.05).float().detach()

            # 2. Edge-Aware Weights berechnen
            with torch.no_grad():
                edge_mag = get_edge_magnitude(img_L_curr)
                # Kanten werden doppelt so stark gewichtet wie Flächen
                edge_w = torch.clamp(edge_mag / (edge_mag.mean() + 1e-6), 0.5, 2.0)

            # 3. Edge-Weighted Photometric Loss
            loss_p_charb_edge = (torch.sqrt((img_L_curr - right_warped)**2 + 1e-6) * mask_vis * edge_w).sum() / (mask_vis.sum() + 1e-6)
            
            grad_l = get_gradients(img_L_curr)
            grad_r_w = get_gradients(right_warped)
            # Edge Weight auch auf Gradient anwenden
            loss_p_grad_edge = (torch.abs(grad_l - grad_r_w) * mask_vis * edge_w).sum() / (mask_vis.sum() + 1e-6)

            # SSIM bleibt Standard (da es ein Window-based Loss ist, ist Pixel-Weighting schwierig)
            s_val = ssim_loss_fn(img_L_curr * mask_vis, right_warped * mask_vis)
            loss_p_ssim = 1.0 - s_val
            
            # Neue Gewichtung
            loss_photo = 0.40 * loss_p_ssim + 0.30 * loss_p_charb_edge + 0.30 * loss_p_grad_edge
            
            total_loss += w_photo * weight * loss_photo
            if i == 0: logs['photo'] = loss_photo.item()
            
            # --- V9.3: Masked Smoothness Loss ---
            # Wir wollen Smoothness NUR auf flachen Flächen erzwingen, NICHT an Kanten
            with torch.no_grad():
                # Flat Mask: 1 wo flach, 0 wo Kante
                flat_mask = (edge_mag < edge_mag.mean()).float()
            
            # Hier wenden wir die Flat Mask an
            loss_s = (smoothness_loss_adaptive(disp_L, img_L_curr) * flat_mask).mean()
            total_loss += w_smooth * weight * loss_s # Smoothness jetzt auf allen Scales gewichtet!

    # --- 3. Okklusions-Loss (nur auf höchster Auflösung) ---
    if "RL" in outputs:
        # Full-Res Maske für Occ Head Supervision
        disp_L_full, disp_R_full = outputs["LR"][0], outputs["RL"][0]
        
        # Grid Full
        grid_y, grid_x = torch.meshgrid(torch.arange(H_full, device=left_img.device), 
                                        torch.arange(W_full, device=left_img.device), indexing='ij')
        grid_x_norm = (2.0 * grid_x / (W_full - 1)) - 1.0
        grid_y_norm = (2.0 * grid_y / (H_full - 1)) - 1.0
        
        vgrid_full = torch.stack((grid_x_norm - (2.0*disp_L_full/(W_full-1)).squeeze(1), 
                                  grid_y_norm.expand(disp_L_full.shape[0],H_full,W_full)), dim=3)
        disp_R_warped_full = F.grid_sample(disp_R_full, vgrid_full, align_corners=False, padding_mode='border')
        lrc_diff_full = torch.abs(disp_L_full - disp_R_warped_full)
        mask_vis_full = (lrc_diff_full < 1.2).float().detach()

        # Supervision für den Occ Head
        if len(outputs["LR"]) > 3: # Sicherstellen, dass occ_logits existieren
            loss_o = F.binary_cross_entropy_with_logits(outputs["LR"][3], 1.0 - mask_vis_full)
            total_loss += w_occ * loss_o
            logs['occ'] = loss_o.item()
    
    return total_loss, logs

In [ ]:
# --- 1. Externer Warper (NUR für Training & Loss) ---
class TrainingWarper(nn.Module):
    def __init__(self):
        super().__init__()
        self.grid_cache = {}
    
    def forward(self, img, disp):
        B, _, H, W = img.shape
        device = img.device
        key = (H, W, device)
        
        if key not in self.grid_cache:
            y, x = torch.meshgrid(torch.arange(H, device=device), torch.arange(W, device=device), indexing='ij')
            self.grid_cache[key] = ( (2.0*x/(W-1))-1.0, (2.0*y/(H-1))-1.0 )
            
        grid_x, grid_y = self.grid_cache[key]
        
        # Warping Vektor: Pixel (u, v) -> (u - disp, v)
        disp_norm = 2.0 * disp / (W - 1)
        vgrid = torch.stack([grid_x - disp_norm.squeeze(1), grid_y.expand(B,H,W)], dim=3)
        
        return F.grid_sample(img, vgrid, align_corners=False, padding_mode='border')

# --- 2. Deine Debug- & Save-Funktionen ---

def get_full_res_preds(model, dataset, index, device='cuda'):
    """Interne Hilfsfunktion: Holt Modell-Output und skaliert ALLES auf 640x480."""
    model.eval()
    with torch.no_grad():
        l, r, dl, dr = dataset[index]
        l_in = l.unsqueeze(0).to(device)
        r_in = r.unsqueeze(0).to(device)
        
        # 1. Forward Pass
        disp_final, disp_v1, disp_low, occ_logits = model.core(l_in, r_in)
        
        # 2. Skalierungs-Logik (bringt alles auf 640x480)
        def up(t):
            if t is None: return None
            curr_w = t.shape[-1]
            scale = 640 / curr_w
            return F.interpolate(t, size=(480, 640), mode='bilinear', align_corners=False) * scale

        # Occ-Logits brauchen keine wertmäßige Skalierung, nur Auflösung
        occ_prob = torch.sigmoid(F.interpolate(occ_logits, size=(480, 640), mode='bilinear', align_corners=False))
        
        # 3. RL Pass für Symmetrie/LRC
        l_f, r_f = torch.flip(l_in, [3]), torch.flip(r_in, [3])
        disp_RL_f, _, _, _ = model.core(r_f, l_f)
        disp_RL = torch.flip(up(disp_RL_f), [3])
        
        # 4. Echo-Map (LRC) Berechnung auf Full Res
        disp_LR = up(disp_final)
        B, _, H, W = disp_LR.shape
        grid_x = torch.arange(W, device=device).view(1, 1, 1, W).expand(B, 1, H, W).float()
        grid_y = torch.arange(H, device=device).view(1, 1, H, 1).expand(B, 1, H, W).float()
        x_proj = grid_x - disp_LR
        norm_x = 2.0 * x_proj / (W - 1) - 1.0
        norm_y = 2.0 * grid_y / (H - 1) - 1.0
        grid = torch.stack((norm_x.squeeze(1), norm_y.squeeze(1)), dim=3)
        disp_RL_warped = F.grid_sample(disp_RL, grid, align_corners=False, padding_mode='border')
        echo_map = torch.abs(disp_LR - disp_RL_warped)

        # Alles nach Numpy [H, W]
        def to_np(t): return t.squeeze().cpu().numpy()
        
        return {
            "l": to_np(l), "dl": to_np(dl), 
            "final": to_np(disp_LR), "v1": to_np(up(disp_v1)), "low": to_np(up(disp_low)),
            "occ": to_np(occ_prob), "echo": to_np(echo_map), "rl_warp": to_np(disp_RL_warped)
        }

def save_debug_visuals(model, dataset, epoch, index=6, device='cuda'):
    d = get_full_res_preds(model, dataset, index, device)
    fig, axs = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f"Debug Visuals - Epoche {epoch}", fontsize=16)

    axs[0,0].imshow(d['l'], cmap='gray'); axs[0,0].set_title("Input Left")
    axs[0,1].imshow(d['dl'], cmap='magma', vmin=0, vmax=192); axs[0,1].set_title("Ground Truth")
    
    im_f = axs[0,2].imshow(d['final'], cmap='magma', vmin=0, vmax=192)
    axs[0,2].set_title("Final Prediction (1/1)"); fig.colorbar(im_f, ax=axs[0,2])

    axs[1,0].imshow(d['low'], cmap='magma', vmin=0, vmax=192); axs[1,0].set_title("Stage Low (1/4)")
    axs[1,1].imshow(d['v1'], cmap='magma', vmin=0, vmax=192); axs[1,1].set_title("Stage V1 (1/2)")
    
    # EPE Map
    epe = np.abs(d['dl'] - d['final'])
    epe[(d['dl'] <= 0) | (d['dl'] >= 192)] = 0
    im_e = axs[1,2].imshow(epe, cmap='jet', vmin=0, vmax=10)
    axs[1,2].set_title("EPE Map (Error)"); fig.colorbar(im_e, ax=axs[1,2])

    for ax in axs.flatten(): ax.axis('off')
    plt.tight_layout(); os.makedirs("training_progress", exist_ok=True)
    plt.savefig(f"training_progress/Fused_Vision-AI_debug_ep_{epoch:03d}.png"); plt.close()

def save_preview(model, dataset, name, index=6, device='cuda'):
    d = get_full_res_preds(model, dataset, index, device)
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f"Preview Analysis: {name}", fontsize=16)

    axes[0,0].imshow(d['l'], cmap='gray'); axes[0,0].set_title("Input")
    axes[0,1].imshow(d['dl'], cmap='magma', vmin=0, vmax=192); axes[0,1].set_title("GT")
    im_p = axes[0,2].imshow(d['final'], cmap='magma', vmin=0, vmax=192)
    axes[0,2].set_title("Prediction"); fig.colorbar(im_p, ax=axes[0,2])

    im_ec = axes[1,0].imshow(d['echo'], cmap='hot', vmin=0, vmax=5)
    axes[1,0].set_title("Echo-Map (LRC Error)"); fig.colorbar(im_ec, ax=axes[1,0])

    im_oc = axes[1,1].imshow(d['occ'], cmap='gray', vmin=0, vmax=1)
    axes[1,1].set_title("Predicted Occlusion"); fig.colorbar(im_oc, ax=axes[1,1])

    epe = np.abs(d['dl'] - d['final'])
    epe[(d['dl'] <= 0) | (d['dl'] >= 192)] = 0
    im_ep = axes[1,2].imshow(epe, cmap='jet', vmin=0, vmax=10)
    axes[1,2].set_title("EPE Error Map"); fig.colorbar(im_ep, ax=axes[1,2])

    for ax in axes.flatten(): ax.axis('off')
    plt.tight_layout(); os.makedirs("training_progress", exist_ok=True)
    plt.savefig(f"training_progress/Fused_Vision-AI_preview_{name}.png"); plt.close()

def plot_disparity_profile(model, dataset, epoch, index=0, device='cuda'):
    d = get_full_res_preds(model, dataset, index, device)
    H, W = d['final'].shape
    rows = [int(H * 0.25), int(H * 0.50), int(H * 0.75)]
    labels = ["25%", "50%", "75%"]
    
    fig, axs = plt.subplots(4, 1, figsize=(12, 16))
    axs[0].imshow(d['l'], cmap='gray')
    for r in rows: axs[0].axhline(r, color='yellow', linewidth=2)
    axs[0].set_title(f"Profile Lines (Epoch {epoch})"); axs[0].axis('off')

    for i, (r, lbl) in enumerate(zip(rows, labels)):
        ax = axs[i+1]
        ax.plot(d['dl'][r, :], 'k-', label='Ground Truth', linewidth=2)
        ax.plot(d['final'][r, :], 'r-', label='Final Prediction', alpha=0.8)
        ax.plot(d['v1'][r, :], 'g--', label='V1 Coarse', alpha=0.6)
        ax.set_title(f"Profile at {lbl} Height (y={r})")
        ax.set_ylim(-5, 200); ax.grid(True, alpha=0.3)
        if i==0: ax.legend()

    plt.tight_layout(); os.makedirs("training_progress", exist_ok=True)
    plt.savefig(f"training_progress/Fused_Vision-AI_profile_ep{epoch:03d}.png"); plt.close()

def save_symmetry_comparison(model, dataset, device, epoch=0, index=0):
    d = get_full_res_preds(model, dataset, index, device)
    fig, axs = plt.subplots(2, 2, figsize=(12, 8))
    
    im1 = axs[0,0].imshow(d['final'], cmap='magma', vmin=0, vmax=192)
    axs[0,0].set_title("LR Prediction"); fig.colorbar(im1, ax=axs[0,0])
    
    im2 = axs[0,1].imshow(d['rl_warp'], cmap='magma', vmin=0, vmax=192)
    axs[0,1].set_title("RL Prediction (Warped to Left)")
    
    im3 = axs[1,0].imshow(d['echo'], cmap='hot', vmin=0, vmax=10)
    axs[1,0].set_title("LRC Difference"); fig.colorbar(im3, ax=axs[1,0])
    
    axs[1,1].imshow(d['l'], cmap='gray'); axs[1,1].set_title("Left Image Input")

    for ax in axs.flat: ax.axis('off')
    plt.tight_layout(); os.makedirs("training_progress", exist_ok=True)
    plt.savefig(f"training_progress/Fused_Vision-AI_symmetry_ep{epoch:03d}.png"); plt.close()

def save_multitask_visuals(model, dataset, teachers, epoch, device, index=6, prefix="vis"):
    """
    Erzeugt Visualisierung inkl. Klassennamen im YOLO-Grid.
    """
    model.eval()
    
    # 1. Daten holen
    if index >= len(dataset): index = 0
    left, right, _ = dataset[index]
    left = left.unsqueeze(0).to(device)
    right = right.unsqueeze(0).to(device)
    
    # 2. Forward Student
    prev_state = model.enable_aux_tasks
    model.enable_aux_tasks = True 
    
    with torch.no_grad():
        preds = model(left, right)
        disp_pred = preds[0]
        seg_logits = preds[4] 
        yolo_feats = preds[5] # List of [1, C, H, W]
    
    model.enable_aux_tasks = prev_state
    
    # 3. Forward Teacher (Targets)
    with torch.no_grad():
        # Jetzt erwarten wir 3 Rückgabewerte!
        seg_target, obj_target, cls_target = teachers.generate_targets(left)
        
    # 4. Daten aufbereiten (Tensor -> Numpy)
    img_vis = left[0, 0].cpu().numpy() * 0.5 + 0.5
    disp_vis = disp_pred[0, 0].cpu().numpy()
    
    # Segmentation
    cmap_seg = ListedColormap(['#666666', '#ff4444', '#4444ff', '#000000'])
    if seg_logits is not None:
        seg_vis = torch.argmax(seg_logits, dim=1)[0].cpu().numpy()
    else:
        seg_vis = np.zeros_like(img_vis)
    seg_target_vis = seg_target[0].cpu().numpy()
    
    # --- YOLO VISUALIZATION (GRID STYLE) ---
    
    # A. Student Predictions (P5 Grid)
    if yolo_feats is not None:
        # Objectness (Sigmoid)
        pred_obj_grid = torch.sigmoid(yolo_feats[2][0, 4]).cpu().numpy()
        # Class (Argmax)
        pred_cls_grid = torch.argmax(yolo_feats[2][0, 5:], dim=0).cpu().numpy()
    else:
        pred_obj_grid = np.zeros((15, 20))
        pred_cls_grid = np.zeros((15, 20))

    # B. Teacher Targets
    # Targets sind schon im Grid-Format
    target_obj_grid = obj_target[0, 0].cpu().numpy()
    target_cls_grid = cls_target[0].cpu().numpy()
    
    # 5. Plotting
    fig, axs = plt.subplots(3, 2, figsize=(12, 14)) # Etwas breiter für Text
    fig.suptitle(f"Multi-Task Inspection | Epoch {epoch}", fontsize=14)
    
    # Row 1: Image & Stereo
    axs[0, 0].imshow(img_vis, cmap='gray')
    axs[0, 0].set_title("Input Left")
    axs[0, 0].axis('off')
    
    axs[0, 1].imshow(disp_vis, cmap='plasma')
    axs[0, 1].set_title("Student Disparity")
    axs[0, 1].axis('off')
    
    # Row 2: Segmentation
    axs[1, 0].imshow(seg_vis, cmap=cmap_seg, vmin=0, vmax=3, interpolation='nearest')
    axs[1, 0].set_title("Student Segmentation")
    axs[1, 0].axis('off')
    
    axs[1, 1].imshow(seg_target_vis, cmap=cmap_seg, vmin=0, vmax=3, interpolation='nearest')
    axs[1, 1].set_title("Teacher Target (Seg)")
    axs[1, 1].axis('off')
    
    # Row 3: YOLO Grid mit Text-Overlay
    # Helper Funktion zum Plotten von Grid + Text
    def plot_grid_with_text(ax, heatmap, classes, threshold=0.3, title=""):
        im = ax.imshow(heatmap, cmap='inferno', vmin=0, vmax=1)
        ax.set_title(title)
        ax.axis('off')
        
        # Grid Dimensionen
        H, W = heatmap.shape
        for y in range(H):
            for x in range(W):
                val = heatmap[y, x]
                cls_id = classes[y, x]
                
                # Zeige Text nur, wenn "Objectness" hoch genug ist
                # oder im Target-Fall, wenn Klasse != -1
                show_text = False
                if title.startswith("Teacher") and cls_id != -1:
                    show_text = True
                elif title.startswith("Student") and val > threshold:
                    show_text = True
                    
                if show_text:
                    if 0 <= cls_id < len(COCO_CLASSES):
                        label = COCO_CLASSES[int(cls_id)]
                        # Kürze lange Namen für Lesbarkeit
                        if len(label) > 6: label = label[:5] + "."
                    else:
                        label = "?"
                    
                    # Textfarbe anpassen (Weiß auf dunklem Grund, Schwarz auf hellem)
                    text_color = 'black' if val > 0.7 else 'white'
                    ax.text(x, y, label, ha='center', va='center', 
                            fontsize=7, color=text_color, fontweight='bold')

    plot_grid_with_text(axs[2, 0], pred_obj_grid, pred_cls_grid, threshold=0.3, title="Student YOLO (P5 Grid)")
    plot_grid_with_text(axs[2, 1], target_obj_grid, target_cls_grid, threshold=0.1, title="Teacher YOLO (Target)")
    
    plt.tight_layout()
    os.makedirs("training_progress", exist_ok=True)
    filename = f"training_progress/Fused_Vision-AI_{prefix}_multitask_ep{epoch:03d}.png"
    plt.savefig(filename)
    plt.close()
    print(f"📸 Debug-Bild gespeichert: {filename}")

In [ ]:
import torch.optim as optim
from tqdm.auto import tqdm
import os
import time

def get_phase_config(epoch):
    config = {}
    
    # --- PHASE 1: GEOMETRY BOOTCAMP (FlyingThings3D) ---
    # Epoche 0 bis 90
    if epoch < 90:
        config['phase'] = "STEREO_BASE"
        config['dataset'] = 'sceneflow'
        config['lr'] = 2e-4 if epoch < 70 else 5e-5 # LR Drop
        config['freeze_backbone'] = False
        config['freeze_stereo'] = False
        config['freeze_seg'] = True # Seg/Yolo aus
        # Gewichte: Nur Stereo
        config['weights'] = {'geom': 1.0, 'photo': 0.2, 'seg': 0.0, 'yolo': 0.0}
        
    # --- PHASE 2: SYNERGY & REALISM (TartanAir) ---
    # Epoche 90 bis 110
    elif epoch < 110:
        config['phase'] = "MULTI_TASK"
        config['dataset'] = 'tartanair'
        config['lr'] = 2e-5 # Vorsichtiges Fine-Tuning
        config['freeze_backbone'] = False # Backbone darf sich leicht anpassen
        config['freeze_stereo'] = False
        config['freeze_seg'] = False
        # Gewichte: Alles an!
        config['weights'] = {'geom': 1.0, 'photo': 0.5, 'seg': 0.5, 'yolo': 0.5}

    # --- PHASE 3: OBJECT SPECIALIST (COCO) ---
    # Epoche 110+ (Fine-Tuning Yolo Head)
    else:
        config['phase'] = "YOLO_FINETUNE"
        config['dataset'] = 'coco'
        config['lr'] = 1e-4 # Head darf schneller lernen
        
        # WICHTIG: Alles außer Yolo einfrieren!
        config['freeze_backbone'] = True
        config['freeze_stereo'] = True
        config['freeze_seg'] = True 
        
        # Gewichte: Nur Yolo
        config['weights'] = {'geom': 0.0, 'photo': 0.0, 'seg': 0.0, 'yolo': 1.0}
        
    return config

def apply_freeze_logic(model, config):
    # 1. Grundsätzlich alles freischalten
    for p in model.parameters():
        p.requires_grad = True
    
    # 2. Spezifische Backbone Stages einfrieren
    for stage_num in config['freeze_backbone_stages']:
        stage = getattr(model.backbone, f'stage{stage_num}')
        for p in stage.parameters():
            p.requires_grad = False
            
    # 3. Stereo Heads einfrieren
    if config['freeze_stereo_heads']:
        for m in [model.cost_reducer, model.spatial_filter, model.refine_final]:
            for p in m.parameters():
                p.requires_grad = False
                
    # 4. Aux-Task Switch
    model.enable_aux_tasks = config['enable_aux_tasks']

# --- Helper Funktion für Gradient Clipping Logging ---
def get_grad_norm(model):
    total_norm = 0
    for p in model.parameters():
        if p.grad is not None:
            param_norm = p.grad.data.norm(2)
            total_norm += param_norm.item() ** 2
    return total_norm ** 0.5

# --- UNIFIED TRAINING LOOP MIT EXTENDED LOGGING ---

def train_strategic_v10(
    model, 
    dataset_paths, 
    start_epoch=0, 
    resume_checkpoint_path=None,
    total_epochs=125, 
    lr_max=2e-4, 
    weight_decay=1e-5,
    accumulation_steps=1, 
    batch_size=4, 
    num_workers=4
):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"🚀 V10 MULTI-TASK STRATEGY START | Device: {device}")
    
    # --- SETUP ---
    scaler = GradScaler()
    warper = TrainingWarper().to(device)
    teachers = DistillationTeachers(device) 
    optimizer = optim.AdamW(model.parameters(), lr=lr_max, weight_decay=weight_decay)
    
    # Checkpoint Loading (dein bestehender Code...)
    # ... (Lade-Logik hier einfügen) ...

    current_dataset_name = ""
    loader_train = None
    loader_val = None
    
    crit_seg = torch.nn.CrossEntropyLoss(ignore_index=3)
    crit_obj = torch.nn.BCEWithLogitsLoss()
    crit_cls = torch.nn.CrossEntropyLoss(ignore_index=-1)

    # ===================== EPOCH LOOP =====================
    for epoch in range(start_epoch, total_epochs):
        
        # --- A. STRATEGIC PHASE CONFIGURATION ---
        # Nutzt deine definierte Logik für Phasen A, B, C, D
        cfg = get_phase_config(epoch)
        
        # Manuelle Overrides für V9-Kompatibilität innerhalb PHASE_A (0-90)
        if cfg['phase'] == "PHASE_A":
            if epoch < 5:      sub_phase, clip, w_geom = "DRILL", 1.0, 2.0
            elif epoch < 30:   sub_phase, clip, w_geom = "STABILITY", 2.0, 1.5
            elif epoch < 75:   sub_phase, clip, w_geom = "REFINE", 5.0, 1.0
            else:              sub_phase, clip, w_geom = "SNIPER", 1.0, 1.2
            cfg['weights']['geom'] = w_geom
            cfg['clip_grad'] = clip
        else:
            sub_phase = cfg['phase']
            cfg['clip_grad'] = 1.0

        curr_lr = cfg['lr']
        dataset_key = cfg['dataset']

        # --- B. APPLY SELECTIVE FREEZING ---
        # Hier wird deine neue Logik angewendet!
        apply_freeze_logic(model, cfg)

        # --- C. DATASET & LOADER SWITCH ---
        if dataset_key != current_dataset_name:
            print(f"\n🔄 SWITCHING DATASET: {current_dataset_name} -> {dataset_key}")
            current_dataset_name = dataset_key
            
            if dataset_key == 'sceneflow':
                ds = StereoDataset(dataset_paths['sceneflow'], mode='train')
                ds_v = StereoDataset(dataset_paths['sceneflow'], mode='val')
            elif dataset_key == 'tartanair':
                ds = TartanAirDataset(dataset_paths['tartanair'], mode='train')
                ds_v = TartanAirDataset(dataset_paths['tartanair'], mode='train')
            elif dataset_key == 'coco':
                ds = COCOSimpleDataset(dataset_paths['coco'])
                ds_v = COCOSimpleDataset(dataset_paths['coco'])
            
            loader_train = DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True, drop_last=True)
            loader_val = DataLoader(ds_v, batch_size=1, shuffle=False, num_workers=num_workers, pin_memory=True)
            
        # --- D. OPTIMIZER & MODE UPDATE ---
        for pg in optimizer.param_groups: 
            pg['lr'] = curr_lr
            
        model.train()
        # Falls Backbone gefroren ist, setzen wir ihn explizit in eval() für BatchNorm Stabilität
        if cfg['freeze_backbone_stages'] == [1, 2, 3, 4] or (epoch >= 110):
            model.backbone.eval()

        # --- E. TRAINING LOOP ---
        pbar = tqdm(loader_train, desc=f"Ep {epoch+1} [{sub_phase}]", ncols=140)
        logs = {'geom':0, 'photo':0, 'seg':0, 'yolo':0, 'occ':0}
        count = 0
        epoch_loss = 0
        
        for batch_idx, (left, right, disp_gt) in enumerate(pbar):
            left, right, disp_gt = left.to(device), right.to(device), disp_gt.to(device)
            
            # Teachers Update
            seg_gt, obj_gt, cls_gt = None, None, None
            if cfg['weights']['seg'] > 0 or cfg['weights']['yolo'] > 0:
                with torch.no_grad():
                    seg_gt, obj_gt, cls_gt = teachers.generate_targets(left)
            
            with autocast():
                preds = model(left, right)
                disp_final, _, _, occ_logits, seg_out, yolo_out = preds
                
                loss = 0
                w = cfg['weights']
                
                # 1. Stereo Losses
                mask = (disp_gt > 0) & (disp_gt < 192)
                valid_stereo = (mask.sum() > 10) and (dataset_key != 'coco')
                
                if valid_stereo and w['geom'] > 0:
                    l_g = torch.nn.functional.smooth_l1_loss(disp_final[mask], disp_gt[mask])
                    loss += w['geom'] * l_g
                    logs['geom'] += l_g.item()
                    
                    if w['photo'] > 0:
                        recon = warper(right, disp_final)
                        l_p = torch.mean(torch.abs(left - recon))
                        loss += w['photo'] * l_p
                        logs['photo'] += l_p.item()
                        
                    # Occ / LRC Logic (V9 Style)
                    if w['lrc'] > 0 or w['occ'] > 0:
                        # Für LRC bräuchten wir eigentlich einen 2. Pass (Right-Left)
                        # Wir simulieren es hier vereinfacht oder nutzen den Occ-Head
                        if occ_logits is not None:
                            # Pseudo-GT für Occ: Wo ist der Geom-Error hoch?
                            # (Nur möglich da wir GT haben)
                            with torch.no_grad():
                                err = torch.abs(disp_final - disp_gt)
                                occ_mask_gt = (err > 3.0).float()
                            l_occ = torch.nn.functional.binary_cross_entropy_with_logits(occ_logits, occ_mask_gt)
                            loss += w['occ'] * l_occ
                            logs['occ'] = logs.get('occ', 0) + l_occ.item()

                # 2. Seg Loss
                if w['seg'] > 0 and seg_out is not None:
                    if seg_out.shape[-2:] != seg_gt.shape[-2:]:
                        seg_out = torch.nn.functional.interpolate(seg_out, size=seg_gt.shape[-2:], mode='bilinear')
                    l_s = crit_seg(seg_out, seg_gt)
                    loss += w['seg'] * l_s
                    logs['seg'] += l_s.item()
                    
                # 3. Yolo Loss (Updated)
                if w['yolo'] > 0 and yolo_out is not None:
                    # yolo_out[2] ist der P5-Output
                    # Shape: [B, (4 + 1 + 80), H, W]
                    # Kanäle: 0-3=Box, 4=Objectness, 5-84=Klassen
                
                    # 1. Objectness Loss (Wie vorher)
                    pred_obj = yolo_out[2][:, 4:5, :, :] 
                    if pred_obj.shape[-2:] != obj_gt.shape[-2:]:
                        # Resize Targets falls nötig
                        obj_gt_res = torch.nn.functional.interpolate(obj_gt, size=pred_obj.shape[-2:])
                        cls_gt_res = torch.nn.functional.interpolate(cls_gt.float().unsqueeze(1), size=pred_obj.shape[-2:], mode='nearest').long().squeeze(1)
                    else: 
                        obj_gt_res = obj_gt
                        cls_gt_res = cls_gt

                    l_obj = crit_obj(pred_obj, obj_gt_res)
                
                    # 2. NEU: Class Loss
                    # Wir nehmen die Kanäle 5 bis Ende (die 80 Klassen)
                    pred_cls = yolo_out[2][:, 5:, :, :] 
                
                    # Wir lernen Klassen NUR dort, wo der Lehrer auch ein Objekt gesehen hat!
                    # crit_cls ignoriert automatisch alles wo cls_gt_res == -1 ist.
                    l_cls = crit_cls(pred_cls, cls_gt_res)
                
                    # Gesamt YOLO Loss
                    loss += w['yolo'] * (l_obj + 0.5 * l_cls) # Klasse ist Bonus
                
                    logs['yolo'] += (l_obj.item() + l_cls.item())
            
            # Optimization
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['clip_grad'])
            scaler.step(optimizer)
            scaler.update()
            
            epoch_loss += loss.item()
            count += 1
            
            # Pbar String
            vram = torch.cuda.memory_reserved(device) / 1024**3
            p_str = f"VRAM:{vram:.1f}G | L:{loss.item():.2f}"
            if w['geom'] > 0: p_str += f" | G:{logs['geom']/count:.2f}"
            if w['seg'] > 0:  p_str += f" | S:{logs['seg']/count:.2f}"
            if w['yolo'] > 0: p_str += f" | Y:{logs['yolo']/count:.2f}"
            pbar.set_postfix_str(p_str)
            
        # --- E. END OF EPOCH ---
        avg_loss = epoch_loss / max(count, 1)
        avg_logs = {k: v/max(count, 1) for k, v in logs.items()}
        
        # Validation (Stereo Metric nur auf Stereo Datasets sinnvoll)
        val_epe = 0.0
        if dataset_key != 'coco':
            val_epe = validate(model, loader_val, device) # Deine validate funktion
        
        print(f"✅ Ep {epoch+1} Done. Avg Loss: {avg_loss:.4f} | Val EPE: {val_epe:.4f}")
        
        # File Log
        with open(log_file, "a") as f:
            f.write(f"{epoch+1}\t{avg_loss:.4f}\t{val_epe:.4f}\t"
                    f"{avg_logs['geom']:.4f}\t{avg_logs['photo']:.4f}\t"
                    f"{avg_logs['seg']:.4f}\t{avg_logs['yolo']:.4f}\t"
                    f"{curr_lr:.2e}\n")
        
        # Debug Visuals (Nur ein Beispiel aus Val)
        try:
            debug_ds = loader_val.dataset
            save_debug_visuals(model, val_loader.dataset, epoch+1, index=0)
            plot_disparity_profile(model, val_loader.dataset, epoch+1, index=0)
            save_preview(model, val_loader.dataset, f"epoch_{(epoch+1):03d}", index=0)
            save_symmetry_comparison(model, val_loader.dataset, device, epoch=epoch+1, index=0)
        except Exception as e:
            pass # Nicht crashen wegen Visuals
            
        # Save Checkpoints
        torch.save(model.state_dict(), "checkpoint_latest.pth")
        if (epoch+1) in [30, 75, 90, 110, 125]:
            torch.save(model.state_dict(), f"Checkpoint_{sub_phase}_Ep{epoch+1}.pth")

    print("🏆 TRAINING COMPLETE.")

# 1. Das neue V10 Modell initialisieren
model = StereoNet_MultiTask_V10(max_disp=192).to(device)

# 2. Deinen besten V9 Checkpoint laden
checkpoint_path = "FusedBackbone-Stereo_BEST.pth" # Oder dein Pfad

if os.path.exists(checkpoint_path):
    print(f"🔄 Lade Basis-Wissen aus {checkpoint_path}...")
    ckpt = torch.load(checkpoint_path, map_location=device)
    
    # Checkpoint State Dict holen
    if 'model_state_dict' in ckpt:
        state_dict = ckpt['model_state_dict']
    else:
        state_dict = ckpt
        
    # Lade-Magie: strict=False ignoriert fehlende Keys (Yolo/Seg)
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    
    print("✅ Backbone & Stereo erfolgreich geladen!")
    print(f"ℹ️  Neu initialisiert (leere Heads): {missing}") 
    # 'missing' sollte keys wie 'seg_head...' und 'yolo_head...' enthalten
else:
    print("⚠️ Kein Checkpoint gefunden! Starte von Null (nicht empfohlen für Schnelltest).")
# Start
train_strategic_v10(
    model, 
    dataset_paths, # Dict: {'sceneflow': '...', 'tartanair': '...', 'coco': '...'}
    start_epoch=90, 
    resume_checkpoint_path="FusedBackbone-Stereo_BEST.pth",
    # --- Training Hyperparameter ---
    total_epochs=125, 
    lr_max=2e-4, 
    weight_decay=1e-5,
    accumulation_steps=8, # Bei 3080Ti und BS=4 meist 1 ok
    batch_size=4, 
    num_workers=4
)